# Principal Component Analysis (PCA)

Throughout, let

$$
X \in \mathbb{R}^{N\times D}
$$

be the data matrix. Rows are observations and columns are variables.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numpy.linalg import svd, norm
from IPython.display import display

np.set_printoptions(precision=4, suppress=True)

PCA is a dimensionality reduction technique.

The **first goal** is to replace variables

$$
X_1,\ldots,X_D
$$

with new ones

$$
Z_1,\ldots,Z_q,
$$

where

$$
q \ll D.
$$

We will call these $Z_i$ the **principal component scores**.

The **second goal** is to avoid losing too much "information".

The central dogma of PCA is that:

$$
\text{variance} \approx \text{information}.
$$

So PCA tries to find new variables with large variance.

When is this reasonable? Example:

In [ ]:
import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(657677)
n = 600

Z = rng.normal(size=(n, 3), scale=1)
Z[:, 0] *= 3.0
Z[:, 1] *= 1.2
Z[:, 2] *= 0.5

theta = np.pi / 5
phi = np.pi / 7

Rz = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0],
    [0,              0,             1]
])

Ry = np.array([
    [ np.cos(phi), 0, np.sin(phi)],
    [0,            1, 0],
    [-np.sin(phi), 0, np.cos(phi)]
])

R = Rz @ Ry
X = Z @ R.T

# Mean-center
xbar = X.mean(axis=0)
X_centered = X - xbar


U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
V = Vt.T

scores = X_centered @ V
X_projected_3d = scores[:, :2] @ V[:, :2].T + xbar

In [ ]:
# Make a grid in PC1-PC2 coordinates
grid_size = 25
pc1_grid = np.linspace(scores[:, 0].min(), scores[:, 0].max(), grid_size)
pc2_grid = np.linspace(scores[:, 1].min(), scores[:, 1].max(), grid_size)

A, B = np.meshgrid(pc1_grid, pc2_grid)

plane_scores = np.column_stack([
    A.ravel(),
    B.ravel()
])

plane_3d = plane_scores @ V[:, :2].T + xbar

PX = plane_3d[:, 0].reshape(grid_size, grid_size)
PY = plane_3d[:, 1].reshape(grid_size, grid_size)
PZ = plane_3d[:, 2].reshape(grid_size, grid_size)

lines = []
scale = 3.5

for j in range(2):
    direction = V[:, j]
    endpoint_1 = xbar - scale * S[j] / np.sqrt(n) * direction
    endpoint_2 = xbar + scale * S[j] / np.sqrt(n) * direction

    lines.append(
        go.Scatter3d(
            x=[endpoint_1[0], endpoint_2[0]],
            y=[endpoint_1[1], endpoint_2[1]],
            z=[endpoint_1[2], endpoint_2[2]],
            mode="lines",
            line=dict(width=8),
            name=f"PC{j + 1}"
        )
    )

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X[:, 0],
        y=X[:, 1],
        z=X[:, 2],
        mode="markers",
        marker=dict(size=3, opacity=0.35),
        name="Original 3D data"
    )
)

fig.add_trace(
    go.Scatter3d(
        x=X_projected_3d[:, 0],
        y=X_projected_3d[:, 1],
        z=X_projected_3d[:, 2],
        mode="markers",
        marker=dict(size=3, opacity=0.35),
        name="Projected onto PCA plane"
    )
)

fig.add_trace(
    go.Surface(
        x=PX,
        y=PY,
        z=PZ,
        opacity=0.25,
        showscale=False,
        name="PCA plane"
    )
)

for line in lines:
    fig.add_trace(line)

fig.update_layout(
    title="Interactive 3D pancake cloud and projection onto the first two PCs",
    scene=dict(
        xaxis_title="$X_1$",
        yaxis_title="$X_2$",
        zaxis_title="$X_3$",
        aspectmode="data"
    ),
    width=900,
    height=700,
    legend=dict(
        x=0.02,
        y=0.98
    )
)

fig.show()

Here is it projected:

In [ ]:
import matplotlib.pyplot as plt

scores_2d = scores[:, :2]

plt.figure(figsize=(7, 6))

plt.scatter(
    scores_2d[:, 0],
    scores_2d[:, 1],
    s=14,
    alpha=0.55
)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.xlabel("$z_1 = X w_1$")
plt.ylabel("$z_2 = X w_2$")
plt.title("Projected data in the first two principal component directions")

plt.gca().set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

Here, we are visualizing $X$ row-wise so that the $n^{\text{th}}$ row is

$$
x_n \in \mathbb{R}^D.
$$

PCA tries to keep this simple, it looks at **linear combinations** of the original variables $X_1,\ldots,X_D$ to create $Z_1,\ldots,Z_q$. Equivalently, the goal is to find a lower-dimensional (linear) **subspace** of $\mathbb{R}^D$ so that when we project the data onto this subspace we don't lose too much information. 

## Projection matrices

Let's do a refresher on projection matrices. Let $W$ be the $D\times q$ matrix of basis elements for the lower-dimensional subspace. 

The projection matrix onto $\operatorname{Col}(W)$ is

$$
P_W = W(W^T W)^{-1}W^T \in \mathbb{R}^{D\times D}.
$$

If $x\in\mathbb{R}^{1 \times D}$ is a row-vector, then

$$
xP_W \in\mathbb{R}^D
$$

is the projected point, still written in the original $\mathbb{R}^D$ coordinates.

For this lecture, WLOG, we can assume that the columns of $W$ are orthonormal so that we have an orthonormal basis (Why not?). In this case, 

$$
W^TW=I_q,
$$

so

$$
P_W=WW^T.
$$

Then

$$
xP_W=xWW^T.
$$

If we want to project all of the data points (rows) in $X\in\mathbb{R}^{N\times D}$ onto $\operatorname{Col}(W)$ then we can write it as: 

$$
XP_W = XWW^T \in\mathbb{R}^{N \times D}
$$

This would be our original data points but now projected down onto the $q$-dimensional subspace. 

There are two related objects:

$$
Z = XW \in \mathbb{R}^{N\times q}
$$

is the data expressed in the new $q$-dimensional coordinates, while

$$
XP_W = XWW^T \in \mathbb{R}^{N\times P}
$$

is the projected data written back in the original $D$ coordinates.

## Principal components as linear combinations

Since PCA wants to find positions of the data points in the lower-dimensional space (i.e using only $q$ variables), then we are mostly interested in 
$$
Z=XW
$$
i.e., the data matrix embedded in the lower-dimensional coordinates.

Notice that if $Z_i$ is the $i^{\text{th}}$ column (variable) of $Z$, $X_j$ is the $j^{\text{th}}$ column (variable) of $X$, and $w_i$ is the $i^{\text{th}}$ column of $W$, then

$$
Z_i = Xw_i
$$

and therefore

$$
Z_i = X_1w_{i1}+X_2w_{i2}+\cdots+X_Pw_{iD}.
$$

So, *equivalently*, each principal component is a linear combination of the original columns of $X$.

## PCA objective

With all this in mind, one way to frame PCA is: it finds linear combinations of the columns (variables) of $X$ such that

1. the variances of the resulting $Z_i$ variables are as large as possible (recall: want to retain information $\approx$ variance)
2. the $Z_i$ variables are uncorrelated (so there is no redundancy)
3. the $w_i$ vectors have unit length (a practical constraint, otherwise variance could grow without bound)

For example, to find the first PC we could sweep over all possible vectors:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(657677)
n = 300

# Start with independent coordinates with unequal variances
Z = rng.normal(size=(n, 2))
Z[:, 0] *= 3.0
Z[:, 1] *= 0.8

# Rotate the cloud so the main direction is not axis-aligned
theta = np.pi / 5

R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

X = Z @ R.T

# Mean-center the data before applying PCA
X = X - X.mean(axis=0)

angles = np.linspace(0, np.pi, 361)

variances = []

for a in angles:
    w_a = np.array([np.cos(a), np.sin(a)])
    z_a = X @ w_a
    variances.append(np.var(z_a, ddof=1))

variances = np.array(variances)

best_angle = angles[np.argmax(variances)]
best_w = np.array([np.cos(best_angle), np.sin(best_angle)])

print("Best direction angle, degrees:", np.rad2deg(best_angle).round(2))
print("Best unit vector:", best_w.round(4))
print("Max projected variance:", variances.max().round(4))

plt.figure(figsize=(6, 4))

plt.plot(np.rad2deg(angles), variances)

plt.xlabel("direction angle in degrees")
plt.ylabel(r"sample variance of $Xw$")
plt.title("PCA chooses the direction with maximal projected variance")

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 6))

plt.scatter(X[:, 0], X[:, 1], s=18, alpha=0.45)

# Draw the best one-dimensional PCA direction
line_t = np.linspace(-4, 4, 100)
line = line_t[:, None] * best_w[None, :]

plt.plot(line[:, 0], line[:, 1], linewidth=3, label="best direction")

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Data cloud and variance-maximizing direction")
plt.gca().set_aspect("equal", adjustable="box")
plt.legend()

plt.tight_layout()
plt.show()

**Ok, so how do we do this generally?**

## Aside: variance and covariance notation

Let $x\in\mathbb{R}^N$ be a variable observed on $N$ units. (**Note** this is a *column* of $X$ here)

Assume

$$
\bar{x}=\frac{1}{N}\sum_{n=1}^N x_n=0.
$$
(if not, mean-center the variable). Then

$$
\widehat{\operatorname{Var}}(x)
=\frac{1}{N-1}\sum_n (x_n-\bar{x})^2
=\frac{1}{N-1}\sum_n x_n^2
\propto x^Tx.
$$

Similarly, if $y\in\mathbb{R}^N$ is another centered variable, then

$$
\widehat{\operatorname{Cov}}(x,y)
=\frac{1}{N-1}x^Ty \propto x^Ty
$$

So for centered variables:

$$
\text{uncorrelated} \quad \Longleftrightarrow \quad \text{orthogonal columns}.
$$

If $X$ is an $N\times D$ data matrix whose columns are mean-centered, then

$$
S = \widehat{\operatorname{Cov}}(X)
=\frac{1}{N-1}X^TX \propto X^TX
$$

This is the covariance matrix, a $D\times D$ matrix, and

$$
S_{ij}=\widehat{\operatorname{Cov}}(X_i,X_j),
\qquad
S_{ii}=\widehat{\operatorname{Var}}(X_i).
$$

## The PCA Problem

Let $X \in \mathbb{R}^{N \times D}$ be centered, so each column has sample mean zero. Define the sample covariance matrix

$$
S_X = \frac{1}{N-1}X^TX.
$$

For a unit direction $w \in \mathbb{R}^D$, the corresponding score variable is

$$
z = Xw.
$$

Its sample variance is

$$
\operatorname{Var}(Xw)=(Xw)^T(Xw) = w^TX^TXw = w^TS_Xw.
$$

Thus PCA can be understood as finding directions $w_1,w_2,\dots$ so that the score variables $Xw_1,Xw_2,\dots$ have large variance and are uncorrelated.

More explicitly, the first principal component is defined by the optimization problem

$$
w_1=\arg\max_{\|w\|_2=1}
\operatorname{Var}(Xw).
$$

Using the formula above, this is equivalent to

$$
w_1=\arg\max_{\|w\|_2=1}
w^T S_X w.
$$

For later components, we want the new score to have large variance while being uncorrelated with the previous scores. So, for $k \geq 2$,

$$
w_k=\arg\max_{\|w\|_2=1,\; \operatorname{Cov}(Xw, Xw_j)=0 \text{ for } j<k}
\operatorname{Var}(Xw).
$$

Equivalently,

$$
w_k=\arg\max_{\|w\|_2=1,\; \operatorname{Cov}(Xw, Xw_j)=0 \text{ for } j<k}
w^T S_X w.
$$

So the objective is sequential: each new component maximizes variance subject to unit length and zero covariance with all previously chosen score variables. Since $X$ is centered,

$$
\operatorname{Cov}(Xw, Xw_j)=w^T S_X w_j.
$$

Therefore, the optimization can also be written as

$$
w_k=\arg\max_{\|w\|_2=1,\; w^T S_X w_j=0 \text{ for } j<k}
w^T S_X w.
$$

After the optimization, the **principal component scores** are then

$$
z_k = Xw_k.
$$

### Simplified Rayleigh quotient theorem

Let $A \in \mathbb{R}^{D \times D}$ be symmetric, with eigendecomposition

$$
A = V\Lambda V^T,
$$

where

$$
\Lambda =\operatorname{diag}(\lambda_1,\dots,\lambda_D),
\qquad
\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_D.
$$

Then

$$
\max_{\|w\|_2=1} w^TAw = \lambda_1.
$$

The maximum is achieved by

$$
w = v_1,
$$

where $v_1$ is the eigenvector corresponding to $\lambda_1$.

More generally,

$$
\max_{\|w\|_2=1,\; w \perp v_1,\dots,v_{k-1}} w^TAw=\lambda_k,
$$

and the maximum is achieved by

$$
w = v_k.
$$

**Proof.** To see why, write $w$ in the eigenvector basis:

$$
w = c_1v_1 + c_2v_2 + \cdots + c_Dv_D.
$$

If $\|w\|_2=1$, then

$$
c_1^2 + c_2^2 + \cdots + c_D^2 = 1.
$$

Now

$$
w^TAw=\lambda_1c_1^2 + \lambda_2c_2^2 + \cdots + \lambda_Dc_D^2.
$$

This is a weighted average of the eigenvalues, with weights $c_i^2$. Since the largest eigenvalue is $\lambda_1$,

$$
w^TAw \leq \lambda_1.
$$

The upper bound is achieved by putting all the weight on the first eigenvector, meaning $w=v_1$.

Similarly, if we require

$$
w \perp v_1,\dots,v_{k-1},
$$

then

$$
c_1 = \cdots = c_{k-1} = 0.
$$

Therefore,

$$
w^TAw=\lambda_kc_k^2 + \lambda_{k+1}c_{k+1}^2 + \cdots + \lambda_Dc_D^2
\leq \lambda_k.
$$

The upper bound is achieved by taking

$$
w = v_k.
$$

### Applying the theorem to PCA

The first principal component solves

$$
w_1=\arg\max_{\|w\|_2=1}
w^TS_Xw.
$$

Since $S_X$ is symmetric (and has all positive eigenvalues), write

$$
S_X = V\Lambda V^T,
$$

where

$$
\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_D \geq 0.
$$

By the Rayleigh quotient theorem,

$$
w_1 = v_1.
$$

So the first principal direction is the eigenvector of $S_X$ with the largest eigenvalue.

The second principal component should maximize variance while being uncorrelated with the first score. The covariance between $Xw$ and $Xw_1$ is

$$
\operatorname{Cov}(Xw, Xw_1)=w^TS_Xw_1.
$$

Since $w_1=v_1$ and $S_Xv_1=\lambda_1v_1$,

$$
w^TS_Xw_1=w^TS_Xv_1=\lambda_1 w^Tv_1.
$$

Therefore, if $\lambda_1>0$, the condition

$$
\operatorname{Cov}(Xw, Xw_1)=0
$$

is equivalent to

$$
w \perp v_1.
$$

So the second direction solves

$$
w_2=\arg\max_{\|w\|_2=1,\; w \perp v_1}
w^TS_Xw.
$$

By the restricted Rayleigh quotient theorem,

$$
w_2 = v_2.
$$

Continuing in the same way, the $k$th principal direction solves

$$
w_k=\arg\max_{\|w\|_2=1,\; w \perp v_1,\dots,v_{k-1}}
w^TS_Xw.
$$

Therefore,

$$
w_k = v_k.
$$

Thus the first $q$ PCA directions are

$$
W_q =\begin{bmatrix}
v_1 & v_2 & \cdots & v_q
\end{bmatrix}.
$$

## Checking Score Properties 

The PCA scores are

$$
Z_q = XW_q.
$$

Their covariance matrix is

$$
S_{Z_q}
=\frac{1}{N-1}Z_q^TZ_q
=W_q^TS_XW_q.
$$

(Need to show these are mean-centered). Since $W_q$ contains the first $q$ eigenvectors of $S_X$,

$$
S_{Z_q}
=\operatorname{diag}(\lambda_1,\dots,\lambda_q).
$$

Thus the PCA scores are uncorrelated, and their variances are

$$
\lambda_1,\dots,\lambda_q.
$$

So the eigenvalue derivation gives exactly what PCA wants: the scores are uncorrelated, their variances are as large as possible in order, and the directions are unit vectors.

**Caveats**: If two *eigenvalues are equal*, then the corresponding eigenvectors are not uniquely determined inside that tied eigenspace. Also, *signs* are arbitrary: replacing $v_i$ by $-v_i$ only flips the sign of the corresponding score.

### Connection to the SVD

Now consider the full SVD of the centered data matrix:

$$
X = UDV^T,
$$

where

$$
U \in \mathbb{R}^{N \times N},
\qquad
D \in \mathbb{R}^{N \times D},
\qquad
V \in \mathbb{R}^{D \times D}.
$$

The matrix $D$ is rectangular diagonal. Its nonzero diagonal entries are the singular values of $X$:

$$
\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0,
$$

where

$$
r = \operatorname{rank}(X).
$$

Using the SVD,

$$
X^TX
=(UDV^T)^T(UDV^T)
=VD^TU^TUDV^T.
$$

Since $U^TU=I$,

$$
X^TX
=VD^TDV^T.
$$

Therefore,

$$
S_X
=\frac{1}{N-1}X^TX
=V\left(\frac{D^TD}{N-1}\right)V^T.
$$

The matrix $D^TD$ is a $D \times D$ diagonal matrix:

$$
D^TD
=\operatorname{diag}
\left(
\sigma_1^2,
\sigma_2^2,
\dots,
\sigma_r^2,
0,
\dots,
0
\right).
$$

As we have seen, the columns of $V$ are the eigenvectors of $S_X$. Therefore, the PCA directions are the **right singular vectors** of $X$.

The eigenvalues of $S_X$ are

$$
\lambda_i = \frac{\sigma_i^2}{N-1}.
$$

So the variance of the $i$th principal component is

$$
\operatorname{Var}(Z_i)
=\frac{\sigma_i^2}{N-1}.
$$
again where $\sigma_i$ is the $i^{th}$ singular value of $X$. 

The score matrix is

$$
Z_q = XV_q,
$$

where

$$
V_q =\begin{bmatrix}
v_1 & v_2 & \cdots & v_q
\end{bmatrix}.
$$

Using the SVD,

$$
Z_q
=XV_q
=UDV^TV_q.
$$

Since $V^TV_q$ selects the first $q$ coordinate directions,

$$
Z_q = U_qD_q,
$$

where

$$
U_q =\begin{bmatrix}
u_1 & u_2 & \cdots & u_q
\end{bmatrix}
$$

and

$$
D_q =\operatorname{diag}(\sigma_1,\dots,\sigma_q).
$$

Essentially, the scores are the columns of $U_q$. 

Alternatively: the PCA scores are the left singular vectors scaled by the corresponding singular values:

$$
Z_i = Xv_i = \sigma_i u_i.
$$

## PCA from SVD

**In summary,** PCA can be computed directly from the SVD of the centered data matrix $X$:

$$
X = UDV^T.
$$

The principal directions (weights $W$) are the first $q$ columns of $V$, the score vectors are the first $q$ columns of $UD$, and the variance of the $i$th score is $\sigma_i^2/(N-1)$.

### Guideline: PCA by SVD

So the steps for calculating PCA are:

0. Mean-center the columns of $X$.

1. Calculate the SVD of $X$:

$$
X=UDV^T.
$$

2. Set

$$
W=V_{1:q}.
$$

3. Compute the scores:

$$
Z=XW=XV_{1:q}.
$$

The columns of $W$ are the **loadings** or **principal directions**.

The columns of $Z$ are the **principal components** or **scores**.

In [ ]:
def pca_via_svd(X,q): 
    
    mean = X.mean(axis=0)
    X0 = X - mean

    U, s, Vt = svd(X0, full_matrices=False)
    V = Vt.T

    W = V[:, :q]
    Z = X0 @ W
    

    return {
        "U": U,
        "singular_values": s,
        "components": V,
        "W": W,
        "scores": Z
    }

res = pca_via_svd(X, q=2)
V = res["components"]
s = res["singular_values"]
Z = res["scores"]

print("Right singular vectors V:")
print(V)
print("\nSingular values:", s)
print("\nVariances from SVs:", s**2/(X.shape[0]-1))
print("\nCovariance of Z:")
print(np.cov(Z, rowvar=False, ddof=1))

In [ ]:
# Plot the PCA directions on top of the synthetic data.

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.55)

for j in range(2):
    direction = V[:, j]
    length = 2.0 
    end = length * direction
    plt.arrow(0, 0, end[0], end[1], width=0.03, length_includes_head=True)
    plt.text(end[0] * 1.05, end[1] * 1.05, f"PC{j+1}")

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel(r"$X_1$")
plt.ylabel(r"$X_2$")
plt.title("PCA directions are the high-variance orthogonal axes")
plt.show()

## Total variance and percent captured

One useful metric is the **total variance captured** by the first $q$ PC scores:

$$
\sum_{i=1}^q \widehat{\operatorname{Var}}(z_i)
=\frac{1}{N-1}\sum_{i=1}^q \sigma_i^2.
$$

Alternatively, one can calculate the **percent of variance captured** by the first $q$ PCs is

$$
\frac{\frac{1}{N-1}\sum_{i=1}^q\sigma_i^2}
{\frac{1}{N-1}\sum_{i=1}^D\sigma_i^2}
= \frac{\sum_{i=1}^q\sigma_i^2}{\sum_{i=1}^D\sigma_i^2}.
$$

In [ ]:
vrs = s**2/(X.shape[0]-1)

In [ ]:
explained_table = pd.DataFrame({
    "PC": np.arange(1, len(s) + 1),
    "singular_value": s,
    "variance": vrs,
    "proportion": vrs/np.sum(vrs),
    "cumulative_proportion": np.cumsum(vrs/np.sum(vrs)),
})
display(explained_table)

## Mean-centering matters

If we do not center the columns, the first principal component can mostly describe the location of the data cloud relative to the origin. In that case, $z_1$ may be close to a mean direction rather than the main direction of variation around the mean.

In [ ]:
rng = np.random.default_rng(657677)
n = 500

# A stretched 2D cloud
Z = rng.normal(size=(n, 2))
Z[:, 0] *= 3.0
Z[:, 1] *= 0.5

# Rotate the cloud
theta = np.pi / 5
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

X_centered_true = Z @ R.T

# Add a large mean shift
mu = np.array([-8.0, 6.0])
X_raw = X_centered_true + mu

In [ ]:
#uncentered
U_unc, s_unc, Vt_unc = svd(X_raw, full_matrices=False)
v_uncentered = Vt_unc.T[:, 0]

#centered
X = X_raw - X_raw.mean(axis=0)
U, s, Vt = svd(X, full_matrices=False)
V = Vt.T
v_centered = V[:, 0]

In [ ]:
v_uncentered

In [ ]:
v_centered

In [ ]:
mean_direction = unit(X_raw.mean(axis=0))

print("Absolute dot product: uncentered PC1 with mean direction")
print(abs(v_uncentered @ mean_direction).round(4))

print("\nAbsolute dot product: centered PC1 with mean direction")
print(abs(v_centered @ mean_direction).round(4))

print("\nUncentered PC1:")
print(v_uncentered.round(4))

print("\nCentered PC1:")
print(v_centered.round(4))

print("\nMean direction:")
print(mean_direction.round(4))

# Real data

For a real data example, use the wine data from `sklearn.datasets`.

The variables are measured in different units, so we use PCA on standardized variables. This means PCA is applied to the **correlation structure** rather than the raw **covariance structure**.

The same SVD method applies after standardization.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine_raw = wine.data.to_numpy()
feature_names = wine.feature_names
target = wine.target.to_numpy()
target_names = wine.target_names

wine_df = wine.frame.copy()
display(wine_df.head())
print("Data shape:", X_wine_raw.shape)
print("Number of classes:", len(target_names))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

# Fit PCA to standardized wine variables
wine_pca_pipe = make_pipeline(
    StandardScaler(),
    PCA()
)

In [ ]:
X_wine_scores = wine_pca_pipe.fit_transform(X_wine_raw)

In [ ]:
X_wine_scores.shape

In [ ]:
X_wine_scores[:10,:3]

In [ ]:
# Pull out fitted PCA object
wine_pca = wine_pca_pipe.named_steps["pca"]

In [ ]:
wine_explained = pd.DataFrame({
    "PC": np.arange(1, len(wine_pca.singular_values_) + 1),
    "singular_value": wine_pca.singular_values_,
    "variance": wine_pca.explained_variance_,
    "proportion": wine_pca.explained_variance_ratio_,
    "cumulative_proportion": np.cumsum(wine_pca.explained_variance_ratio_),
})

display(wine_explained.head(10))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(wine_explained["PC"], wine_explained["proportion"], marker="o")
plt.xlabel("principal component")
plt.ylabel("proportion of variance explained")
plt.title("Wine data scree plot")
plt.xticks(wine_explained["PC"])
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(wine_explained["PC"], wine_explained["cumulative_proportion"], marker="o")
plt.xlabel("number of PCs retained")
plt.ylabel("cumulative proportion of variance explained")
plt.title("Cumulative variance explained in the wine data")
plt.xticks(wine_explained["PC"])
plt.ylim(0, 1.05)
plt.show()

In [ ]:
Z_wine = X_wine_scores

plt.figure(figsize=(7, 5))

for k, name in enumerate(target_names):
    mask = target == k
    plt.scatter(
        Z_wine[mask, 0],
        Z_wine[mask, 1],
        alpha=0.75,
        label=name
    )

plt.xlabel("PC1 score")
plt.ylabel("PC2 score")
plt.title("Wine data projected onto the first two PCs")
plt.legend(title="class")

plt.tight_layout()
plt.show()

## Interpreting loadings

The loadings are the entries of the vectors $v_i$.

For PC1,

$$
z_1=Xv_1=X_1v_{11}+X_2v_{12}+\cdots+X_Dv_{1P}.
$$

Variables with large absolute loading values contribute more to that principal component. The sign is meaningful relative to other variables, but the whole vector can be multiplied by $-1$ without changing the PCA solution. Sometimes we can "interpret" what the PCs are telling us:

In [ ]:
pc1_loadings = pd.Series(
    wine_pca.components_[0, :],
    index=feature_names,
    name="PC1 loading"
)

pc2_loadings = pd.Series(
    wine_pca.components_[1, :],
    index=feature_names,
    name="PC2 loading"
)

loading_table = pd.concat([pc1_loadings, pc2_loadings], axis=1)

loading_table["abs PC1 loading"] = loading_table["PC1 loading"].abs()

loading_table = loading_table.sort_values(
    "abs PC1 loading",
    ascending=False
)

display(loading_table)

In [ ]:
loading_table["PC1 loading"].sort_values().plot(kind="barh", figsize=(7, 5))
plt.xlabel("loading value")
plt.title("PC1 loadings for standardized wine variables")
plt.show()

# High-dimensional Visualization: MNIST

We can also use PCA to visualize high dimensional data like MNIST:

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False)

X = mnist.data.astype(float)
y = mnist.target.astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

rng = np.random.default_rng(657677)

n_sample = 10000
idx = rng.choice(X.shape[0], size=n_sample, replace=False)

X_sub = X[idx]
y_sub = y[idx]

# Scale pixels to [0, 1]
X_sub = X_sub / 255.0

In [ ]:
# PCA centers internally, so we do not need to manually subtract the mean.
pca = PCA(n_components=2)
Z = pca.fit_transform(X_sub)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

print("\nTotal variance explained by first 2 PCs:")
print(pca.explained_variance_ratio_.sum().round(4))

In [ ]:
plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    Z[:, 0],
    Z[:, 1],
    c=y_sub,
    s=8,
    alpha=0.75,
    cmap="tab10"
)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.title("MNIST projected onto the first two principal components")

cbar = plt.colorbar(scatter, ticks=np.arange(10))
cbar.set_label("Digit")

plt.tight_layout()
plt.show()